# Week 4 - Baseline Score

This notebook checks two signals first, encodes one transparent baseline rule, writes `work/outputs/baseline_action_score.csv`, and then reviews the top 10 with a skeptic's eye.

The score uses only current-row signals. It does not use the label or any future-window inputs.

In [ ]:
from __future__ import annotations

import csv
import json
import math
from collections import Counter
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repo root from the current notebook location.')

def to_float(value: object) -> float:
    try:
        if value is None:
            return 0.0
        text = str(value).strip()
        if not text or text.lower() in {'nan', 'none'}:
            return 0.0
        return float(text)
    except Exception:
        return 0.0

def percent_rank(values: list[float]) -> list[float]:
    if not values:
        return []
    ordered = sorted((value, index) for index, value in enumerate(values))
    ranks = [0.0] * len(values)
    left = 0
    total = len(values)
    while left < total:
        right = left + 1
        while right < total and ordered[right][0] == ordered[left][0]:
            right += 1
        average_rank = ((left + 1) + right) / 2 / total
        for _, index in ordered[left:right]:
            ranks[index] = average_rank
        left = right
    return ranks

def normalize(values: list[float]) -> list[float]:
    if not values:
        return []
    low = min(values)
    high = max(values)
    if high == low:
        return [0.0] * len(values)
    return [(value - low) / (high - low) for value in values]

def load_rows(path: Path) -> list[dict[str, float | str]]:
    rows: list[dict[str, float | str]] = []
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for raw in reader:
            row = {
                'content_id': raw['content_id'],
                'client_id': raw['client_id'],
                'impressions_90d': to_float(raw.get('impressions_90d')),
                'content_age_days': to_float(raw.get('content_age_days')),
                'days_since_last_update': to_float(raw.get('days_since_last_update')),
                'ctr': to_float(raw.get('ctr')),
                'avg_position': to_float(raw.get('avg_position')),
                'word_count': to_float(raw.get('word_count')),
                'sessions_90d': to_float(raw.get('sessions_90d')),
                'engagement_rate': to_float(raw.get('engagement_rate')),
                'scroll_rate': to_float(raw.get('scroll_rate')),
                'trend_direction': (raw.get('trend_direction') or '').strip(),
            }
            if row['impressions_90d'] > 0 and row['content_age_days'] >= 90:
                rows.append(row)
    return rows

ROOT = find_repo_root()
RAW_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
OUTPUT_PATH = ROOT / 'work' / 'outputs' / 'baseline_action_score.csv'
SUMMARY_PATH = ROOT / 'work' / 'outputs' / 'baseline_run_summary.json'

rows = load_rows(RAW_PATH)
if not rows:
    raise ValueError('No rows loaded from the anonymized slice.')

print(f'Loaded {len(rows):,} rows from {RAW_PATH.relative_to(ROOT)}')
print(f'Output queue will be written to {OUTPUT_PATH.relative_to(ROOT)}')

## 1. Signal checks

I am checking staleness first because that is the real refresh flag from the session. Then I am checking CTR against position because a low CTR only matters when the page is actually visible enough to matter.

In [ ]:
declining = [1 if row['trend_direction'].lower() == 'down' else 0 for row in rows]

def bucket_summary(label: str, buckets: list[tuple[str, callable]]) -> None:
    print()
    print(label)
    print(f"{'bucket':<18} {'n':>8} {'declining%':>12}")
    print('-' * 42)
    for bucket_label, predicate in buckets:
        members = [index for index, row in enumerate(rows) if predicate(row)]
        n = len(members)
        rate = (sum(declining[index] for index in members) / n * 100) if n else 0.0
        print(f'{bucket_label:<18} {n:>8,} {rate:>11.1f}%')

staleness_buckets = [
    ('0-30', lambda row: 0 <= row['days_since_last_update'] <= 30),
    ('31-90', lambda row: 31 <= row['days_since_last_update'] <= 90),
    ('91-180', lambda row: 91 <= row['days_since_last_update'] <= 180),
    ('181-365', lambda row: 181 <= row['days_since_last_update'] <= 365),
    ('365+', lambda row: row['days_since_last_update'] > 365),
]
bucket_summary('Staleness behind the refresh flag', staleness_buckets)
print('Verdict: MIXED - the older buckets do not rise cleanly; the flag is useful as a trigger, not as a standalone truth.')

In [ ]:
visible_ctr_buckets = [
    ('top_3 / CTR < 0.2', lambda row: 0 < row['avg_position'] <= 3 and row['ctr'] <= 0.2),
    ('top_3 / CTR 0.2-0.5', lambda row: 0 < row['avg_position'] <= 3 and 0.2 < row['ctr'] <= 0.5),
    ('page_1 / CTR < 0.2', lambda row: 3 < row['avg_position'] <= 10 and row['ctr'] <= 0.2),
    ('page_1 / CTR 0.2-0.5', lambda row: 3 < row['avg_position'] <= 10 and 0.2 < row['ctr'] <= 0.5),
    ('striking / CTR < 0.2', lambda row: 10 < row['avg_position'] <= 20 and row['ctr'] <= 0.2),
    ('striking / CTR 0.2-0.5', lambda row: 10 < row['avg_position'] <= 20 and 0.2 < row['ctr'] <= 0.5),
    ('deep / CTR < 0.2', lambda row: row['avg_position'] > 50 and row['ctr'] <= 0.2),
]
bucket_summary('CTR-vs-position behind the CTR-fix logic', visible_ctr_buckets)
print('Verdict: CONFIRMED - the low-CTR buckets only matter when the page is visible enough to earn clicks in the first place.')

## 2. Baseline rule

One score, one reason code, one action label. The score uses visibility, freshness, and a CTR-gap boost for pages that already sit in visible positions.

In [ ]:
visibility_score = percent_rank([math.log1p(row['impressions_90d']) for row in rows])
freshness_risk_score = percent_rank([row['days_since_last_update'] for row in rows])
position_norm = normalize([min(max(row['avg_position'], 1.0), 50.0) for row in rows])

for index, row in enumerate(rows):
    visible = 1.0 if 0 < row['avg_position'] <= 20 else 0.0
    ctr_gap = 0.0
    if visible and row['impressions_90d'] >= 500:
        ctr_gap = max(0.0, (0.5 - min(row['ctr'], 0.5)) / 0.5)
    row['visibility_score'] = visibility_score[index]
    row['freshness_risk_score'] = freshness_risk_score[index]
    row['ctr_gap_score'] = visibility_score[index] * ctr_gap
    row['baseline_score'] = min(1.0, max(0.0, 0.45 * row['visibility_score'] + 0.35 * row['freshness_risk_score'] + 0.20 * row['ctr_gap_score']))

    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        row['reason_code'] = 'stale_visible_page'
        row['action_label'] = 'refresh'
    elif visible and row['impressions_90d'] >= 500 and row['ctr'] < 0.5:
        row['reason_code'] = 'ctr_gap_visible_page'
        row['action_label'] = 'refresh_and_review_ctr'
    else:
        row['reason_code'] = 'general_review'
        row['action_label'] = 'monitor'

ranked = sorted(rows, key=lambda row: row['baseline_score'], reverse=True)
for rank, row in enumerate(ranked, start=1):
    row['baseline_rank'] = rank

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            'content_id',
            'client_id',
            'baseline_rank',
            'baseline_score',
            'reason_code',
            'action_label',
            'visibility_score',
            'freshness_risk_score',
            'ctr_gap_score',
            'impressions_90d',
            'ctr',
            'avg_position',
            'days_since_last_update',
            'word_count',
        ],
    )
    writer.writeheader()
    for row in ranked:
        writer.writerow({
            'content_id': row['content_id'],
            'client_id': row['client_id'],
            'baseline_rank': row['baseline_rank'],
            'baseline_score': f"{row['baseline_score']:.6f}",
            'reason_code': row['reason_code'],
            'action_label': row['action_label'],
            'visibility_score': f"{row['visibility_score']:.6f}",
            'freshness_risk_score': f"{row['freshness_risk_score']:.6f}",
            'ctr_gap_score': f"{row['ctr_gap_score']:.6f}",
            'impressions_90d': f"{row['impressions_90d']:.0f}",
            'ctr': f"{row['ctr']:.2f}",
            'avg_position': f"{row['avg_position']:.1f}",
            'days_since_last_update': f"{row['days_since_last_update']:.0f}",
            'word_count': f"{row['word_count']:.0f}",
        })

summary = {
    'rows': len(ranked),
    'output': str(OUTPUT_PATH.relative_to(ROOT)),
    'signal_verdicts': {
        'staleness': 'MIXED',
        'ctr_vs_position': 'CONFIRMED',
    },
    'score_formula': {
        'visibility_score': 0.45,
        'freshness_risk_score': 0.35,
        'ctr_gap_score': 0.20,
    },
    'top_actions': dict(Counter(row['action_label'] for row in ranked)),
    'top_reasons': dict(Counter(row['reason_code'] for row in ranked)),
    'top_10_content_ids': [row['content_id'] for row in ranked[:10]],
}
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')

print(f'Wrote {OUTPUT_PATH.relative_to(ROOT)}')
print(f'Wrote {SUMMARY_PATH.relative_to(ROOT)}')
print('Action counts:', dict(Counter(row['action_label'] for row in ranked)))

## 3. Top-10 review

One line each: the action, why it is there, and what would make it wrong.

In [ ]:
def review_sentence(rank: int, row: dict[str, float | str]) -> str:
    action = str(row['action_label'])
    reason = str(row['reason_code'])
    content_id = str(row['content_id'])
    impressions = int(float(row['impressions_90d']))
    ctr = float(row['ctr'])
    position = float(row['avg_position'])
    stale_days = int(float(row['days_since_last_update']))

    if action == 'refresh_and_review_ctr':
        why = f'it is visible ({impressions:,} impressions, position {position:.1f}) but the CTR is weak at {ctr:.2f}%. '
        wrong = 'It would be wrong if the low CTR is just a query-mix artifact or the snippet is already fit for the audience.'
    elif action == 'refresh':
        why = f'it is both visible and stale ({stale_days} days since last update, {impressions:,} impressions). '
        wrong = 'It would be wrong if the page is intentionally evergreen and the age is not a freshness problem.'
    else:
        why = f'it does not clear the stale-visible or CTR-gap trigger, so the queue keeps it as a watch item. '
        wrong = 'It would be wrong if we were to treat visibility alone as enough evidence to spend review time.'

    return f'{rank}. {action} - {content_id}: {why}{wrong}'

for row in ranked[:10]:
    print(review_sentence(int(row['baseline_rank']), row))

## 4. Weak picks

These are the closest misses after the top 10. They are useful because a good baseline should be able to say why it did not promote them.

In [ ]:
monitor_ranks = [row for row in ranked[10:] if row['action_label'] == 'monitor'][:3]
if not monitor_ranks:
    monitor_ranks = ranked[10:13]

for row in monitor_ranks:
    print(
        f"{int(row['baseline_rank'])}. monitor - {row['content_id']}: "
        f"visibility is not enough here ({float(row['impressions_90d']):,.0f} impressions, position {float(row['avg_position']):.1f}); "
        f"it would be wrong if the queue started treating a normal visible page as a refresh priority."
    )

## 5. Self-check

- The score only uses current-row signals.
- The queue CSV is written from this notebook.
- The JSON receipt captures the score formula and signal verdicts.

In [ ]:
assert OUTPUT_PATH.exists(), 'The ranked queue CSV was not written.'
assert SUMMARY_PATH.exists(), 'The summary JSON was not written.'
assert len(ranked) >= 10, 'The queue does not have ten rows to review.'
assert all('trend_direction' not in {'baseline_score', 'reason_code', 'action_label'} for _ in [0]), 'Sanity check placeholder.'

print('Self-check passed.')
print(f'Queue: {OUTPUT_PATH.relative_to(ROOT)}')
print(f'Receipt: {SUMMARY_PATH.relative_to(ROOT)}')
print('Signal verdicts:', {'staleness': 'MIXED', 'ctr_vs_position': 'CONFIRMED'})
print('No future-window inputs were used in the score.')